# Train CNN on Word Frames (Colab)

This notebook trains the ASL_CNN model on word frames to extract 512-dim features.

## Setup Instructions
1. Upload `words.zip` to your Google Drive (zip the `words/` folder from your repo)
2. Run this notebook with GPU runtime (Runtime > Change runtime type > GPU)
3. Download the trained model at the end


In [ ]:
# Check GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected! Training will be very slow.")
    print("Go to Runtime > Change runtime type > GPU")

## 1. Mount Google Drive & Extract Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Update this path to where you uploaded words.zip in Google Drive
ZIP_PATH = "/content/drive/MyDrive/words.zip"

import os
if not os.path.exists(ZIP_PATH):
    print(f"ERROR: {ZIP_PATH} not found!")
    print("Please update ZIP_PATH to the correct location in your Google Drive")
else:
    print(f"Found: {ZIP_PATH}")
    !ls -lh "{ZIP_PATH}"

In [ ]:
# Extract the data
!unzip -q "{ZIP_PATH}" -d /content/
!ls /content/words/

In [ ]:
# Flatten the nested video structure into ImageFolder format
# words/<split>/<word>/<video_id>/frame_*.jpg -> word_frames_flat/<split>/<word>/<video_id>_frame_*.jpg

import os
import glob
import shutil
from tqdm import tqdm

WORDS_ROOT = "/content/words"
OUTPUT_ROOT = "/content/word_frames_flat"

print("Flattening video frames into ImageFolder structure...")

for split in ["train", "val", "test"]:
    split_dir = os.path.join(WORDS_ROOT, split)
    if not os.path.isdir(split_dir):
        print(f"  {split}: not found, skipping")
        continue
    
    frame_count = 0
    words = sorted(os.listdir(split_dir))
    
    for word in tqdm(words, desc=f"{split}"):
        word_dir = os.path.join(split_dir, word)
        if not os.path.isdir(word_dir):
            continue
        
        # Create output directory for this word class
        output_class_dir = os.path.join(OUTPUT_ROOT, split, word)
        os.makedirs(output_class_dir, exist_ok=True)
        
        # Process each video
        for video_id in os.listdir(word_dir):
            video_dir = os.path.join(word_dir, video_id)
            if not os.path.isdir(video_dir):
                continue
            
            # Get all frames
            frame_files = glob.glob(os.path.join(video_dir, "frame_*.jpg"))
            for frame_path in frame_files:
                frame_name = os.path.basename(frame_path)
                # Create unique name: video_id + frame_name
                unique_name = f"{video_id}_{frame_name}"
                output_path = os.path.join(output_class_dir, unique_name)
                
                # Create symlink (faster than copy)
                if not os.path.exists(output_path):
                    os.symlink(os.path.abspath(frame_path), output_path)
                frame_count += 1
    
    print(f"  {split}: {frame_count} frames")

print("\nFlattening complete!")

In [ ]:
# Check data structure
import os

DATA_ROOT = "/content/word_frames_flat"

for split in ["train", "val", "test"]:
    split_dir = os.path.join(DATA_ROOT, split)
    if os.path.exists(split_dir):
        classes = os.listdir(split_dir)
        total_images = sum(
            len(os.listdir(os.path.join(split_dir, c)))
            for c in classes if os.path.isdir(os.path.join(split_dir, c))
        )
        print(f"{split}: {total_images} images, {len(classes)} classes")

## 2. Define the CNN Model

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
from torchvision.models import ResNet50_Weights


class ASL_CNN(nn.Module):
    '''
    CNN for ASL recognition using transfer learning from ResNet50.
    Outputs 512-dim features before final classification layer.
    '''
    def __init__(self, num_classes=45):
        super(ASL_CNN, self).__init__()

        # Batch normalization as the first layer
        self.batch_norm = nn.BatchNorm2d(3)

        self._base_model = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)

        # Freeze entire ResNet backbone; train only FC head
        for param in self._base_model.parameters():
            param.requires_grad = False

        num_features = self._base_model.fc.in_features  # 2048
        self._base_model.fc = nn.Sequential(
            nn.Linear(num_features, 512),
            nn.ReLU(),
            nn.Dropout(0.6),  # Increased from 0.4 to reduce overfitting
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.batch_norm(x)
        return self._base_model(x)


class ASL_CNN_FeatureExtractor(nn.Module):
    '''
    Feature extractor that outputs 512-dim features.
    '''
    def __init__(self, trained_asl_cnn):
        super(ASL_CNN_FeatureExtractor, self).__init__()

        self.batch_norm = trained_asl_cnn.batch_norm

        self.resnet_layers = nn.Sequential(
            trained_asl_cnn._base_model.conv1,
            trained_asl_cnn._base_model.bn1,
            trained_asl_cnn._base_model.relu,
            trained_asl_cnn._base_model.maxpool,
            trained_asl_cnn._base_model.layer1,
            trained_asl_cnn._base_model.layer2,
            trained_asl_cnn._base_model.layer3,
            trained_asl_cnn._base_model.layer4,
            trained_asl_cnn._base_model.avgpool
        )

        self.fc_layers = nn.Sequential(
            trained_asl_cnn._base_model.fc[0],  # Linear(2048 -> 512)
            trained_asl_cnn._base_model.fc[1],  # ReLU
            trained_asl_cnn._base_model.fc[2]   # Dropout
        )

    def forward(self, x):
        x = self.batch_norm(x)
        x = self.resnet_layers(x)
        x = torch.flatten(x, 1)
        x = self.fc_layers(x)
        return x  # (batch, 512)


print("Model classes defined successfully!")

## 3. Setup Data Loaders

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Image transforms - AGGRESSIVE augmentation to prevent overfitting
# The model was memorizing videos, not learning word features
TRAIN_TRANSFORM = transforms.Compose([
    transforms.Resize((144, 144)),  # Larger for random crop
    transforms.RandomCrop((128, 128)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(20),
    transforms.RandomAffine(degrees=0, translate=(0.15, 0.15), scale=(0.85, 1.15)),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.1),
    transforms.RandomGrayscale(p=0.15),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.15)),
])

VAL_TRANSFORM = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# Load datasets
train_dataset = datasets.ImageFolder(f"{DATA_ROOT}/train", transform=TRAIN_TRANSFORM)
val_dataset = datasets.ImageFolder(f"{DATA_ROOT}/val", transform=VAL_TRANSFORM)
test_dataset = datasets.ImageFolder(f"{DATA_ROOT}/test", transform=VAL_TRANSFORM)

NUM_CLASSES = len(train_dataset.classes)
CLASS_NAMES = train_dataset.classes

print(f"Train: {len(train_dataset)} samples")
print(f"Val:   {len(val_dataset)} samples")
print(f"Test:  {len(test_dataset)} samples")
print(f"\nClasses ({NUM_CLASSES}): {CLASS_NAMES}")

In [ ]:
# Create dataloaders
BATCH_SIZE = 64  # Larger batch size for GPU

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

## 4. Training Functions

In [ ]:
import time
from tqdm import tqdm

def train_one_epoch(model, dataloader, criterion, optimizer, device):
    """Train for one epoch with progress bar."""
    model.train()
    running_loss = 0.0
    running_corrects = 0
    total_samples = 0

    pbar = tqdm(dataloader, desc="Training", leave=False)
    for inputs, labels in pbar:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels.data)
        total_samples += inputs.size(0)

        # Update progress bar
        current_loss = running_loss / total_samples
        current_acc = running_corrects.double() / total_samples
        pbar.set_postfix({"loss": f"{current_loss:.4f}", "acc": f"{current_acc:.4f}"})

    epoch_loss = running_loss / total_samples
    epoch_acc = running_corrects.double() / total_samples
    return epoch_loss, epoch_acc.item()


def validate(model, dataloader, criterion, device):
    """Validate the model."""
    model.eval()
    running_loss = 0.0
    running_corrects = 0
    total_samples = 0

    with torch.no_grad():
        for inputs, labels in tqdm(dataloader, desc="Validating", leave=False):
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)
            total_samples += inputs.size(0)

    epoch_loss = running_loss / total_samples
    epoch_acc = running_corrects.double() / total_samples
    return epoch_loss, epoch_acc.item()

## 5. Train the Model

In [ ]:
# Hyperparameters - tuned to reduce overfitting
NUM_EPOCHS = 30
LEARNING_RATE = 5e-5      # Lower LR for pretrained layers
FC_LEARNING_RATE = 5e-4   # Lower LR for new FC layers
WEIGHT_DECAY = 1e-3       # L2 regularization
TARGET_ACCURACY = 0.90    # More realistic target (was 0.99)

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Initialize model
model = ASL_CNN(num_classes=NUM_CLASSES)
model = model.to(device)

# Loss function with label smoothing to prevent overconfidence
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Optimizer with different learning rates AND weight decay
pretrained_params = []
fc_params = []
for name, param in model.named_parameters():
    if param.requires_grad:
        if 'fc' in name:
            fc_params.append(param)
        else:
            pretrained_params.append(param)

optimizer = torch.optim.AdamW([
    {'params': pretrained_params, 'lr': LEARNING_RATE},
    {'params': fc_params, 'lr': FC_LEARNING_RATE}
], weight_decay=WEIGHT_DECAY)

# Learning rate scheduler - reduce on plateau instead of fixed steps
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=3, verbose=True
)

print(f"Model initialized with {NUM_CLASSES} classes")
print(f"Pretrained params LR: {LEARNING_RATE}")
print(f"FC params LR: {FC_LEARNING_RATE}")
print(f"Weight decay: {WEIGHT_DECAY}")
print(f"Label smoothing: 0.1")

In [ ]:
# Training loop
print("=" * 70)
print(f"Starting training for up to {NUM_EPOCHS} epochs")
print(f"Early stopping at {TARGET_ACCURACY:.0%} validation accuracy")
print("=" * 70)

best_acc = 0.0
best_model_state = None
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

start_time = time.time()

for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")
    print("-" * 40)

    # Train
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)

    # Validate
    val_loss, val_acc = validate(model, val_loader, criterion, device)

    # Step scheduler (ReduceLROnPlateau needs the metric)
    scheduler.step(val_acc)

    # Record history
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")

    # Save best model
    if val_acc > best_acc:
        best_acc = val_acc
        best_model_state = model.state_dict().copy()
        print(f"-> New best model! (Val Acc: {best_acc:.4f})")

    # Early stopping
    if val_acc >= TARGET_ACCURACY:
        print(f"\n*** Target accuracy {TARGET_ACCURACY:.0%} reached! Stopping early. ***")
        break

total_time = time.time() - start_time
print("\n" + "=" * 70)
print("Training Complete!")
print(f"Total time: {total_time/60:.1f} minutes")
print(f"Best Val Accuracy: {best_acc:.4f} ({best_acc*100:.2f}%)")
print("=" * 70)

## 6. Evaluate on Test Set

In [ ]:
# Load best model
model.load_state_dict(best_model_state)

# Evaluate on test set
test_loss, test_acc = validate(model, test_loader, criterion, device)

print("=" * 70)
print("TEST RESULTS")
print("=" * 70)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")
print("=" * 70)

## 7. Save & Download Model

In [ ]:
# Save the model
MODEL_SAVE_PATH = "/content/best_cnn_words.pth"
torch.save(best_model_state, MODEL_SAVE_PATH)
print(f"Model saved to: {MODEL_SAVE_PATH}")

# Also save class names for reference
CLASS_SAVE_PATH = "/content/word_classes.txt"
with open(CLASS_SAVE_PATH, "w") as f:
    for i, cls in enumerate(CLASS_NAMES):
        f.write(f"{i},{cls}\n")
print(f"Class names saved to: {CLASS_SAVE_PATH}")

In [ ]:
# Download the model file
from google.colab import files

print("Downloading model file...")
files.download(MODEL_SAVE_PATH)

print("\nDownloading class names...")
files.download(CLASS_SAVE_PATH)

## 8. (Optional) Copy to Google Drive

In [ ]:
# Alternatively, save to Google Drive for persistence
DRIVE_SAVE_DIR = "/content/drive/MyDrive/LearningASL_models/"

import os
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

!cp {MODEL_SAVE_PATH} {DRIVE_SAVE_DIR}
!cp {CLASS_SAVE_PATH} {DRIVE_SAVE_DIR}

print(f"Model copied to Google Drive: {DRIVE_SAVE_DIR}")
!ls -lh {DRIVE_SAVE_DIR}

---
## Next Steps

After downloading `best_cnn_words.pth`, place it in your local repo:

```
LearningASL/
└── models/
    └── best_cnn_words.pth   <-- Put it here
```

Then continue with the pipeline:
1. Extract CNN features using `ASL_CNN_FeatureExtractor`
2. Extract keypoints using MediaPipe
3. Train the transformer on combined features


In [ ]:
# Quick test: Verify feature extractor works
print("Testing feature extraction...")

feature_extractor = ASL_CNN_FeatureExtractor(model)
feature_extractor = feature_extractor.to(device)
feature_extractor.eval()

# Get a sample batch
sample_batch, _ = next(iter(test_loader))
sample_batch = sample_batch.to(device)

with torch.no_grad():
    features = feature_extractor(sample_batch)

print(f"Input shape: {sample_batch.shape}")
print(f"Output features shape: {features.shape}")
print(f"Expected: (batch_size, 512)")
print("\nFeature extractor works correctly!")